::: {.sb-lesson-links}
[Exercises →](../exercises/titanic.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/01-preprocessing.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-preprocessing-with-pandas.ipynb){download="lesson-preprocessing-with-pandas.ipynb"}  
Data: [dataset_preprocessing.csv](../downloads/data/dataset_preprocessing.csv){download="dataset_preprocessing.csv"} · [titanic.csv](../downloads/data/titanic.csv){download="titanic.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

## What you'll learn in this class 🧐🧐

In this course, you will learn how to do machine learning. The preparation step you need to take before using any machine learning model is called preprocessing. It comes after ETL, the process of taking messy, unstructured data and converting it into a structured, often tabular, form that is fit for further analysis. Preprocessing starts once you have structured data that you need to clean before you can train your models. Your data may be structured, but it can still contain a wide variety of data types (numbers, texts, dates, times...) that you will have to transform. You will also have to deal with missing data, because a machine learning model is a mathematical model that only knows how to process numbers. Here's what you'll be able to do after this course:

* Choose the right preprocessing methods according to the type of data you want to use in a machine learning model

Along the way, small demos 🔬 show each preprocessing step on real data: the passengers of the Titanic. The next cell loads the libraries and the dataset.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"

titanic = pd.read_csv("../data/titanic.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## What is preprocessing 👷👷

From this module onwards, you will get to the heart of the subject of machine learning! Without going into details (this is not the purpose of this course), for now you can remember that "machine learning" is a whole set of methods that make it possible to carry out tasks without explicitly coding all the steps these tasks require. The applications are very diverse: image recognition, semantic analysis, cost prediction, customer segmentation,... The principle of machine learning is to "show" data to a mathematical model, which "learns" from it according to a set of rules that define the optimal way to perform its task.

You may have noticed that in Data Science, we handle different types of data: tables of numbers, but also text, or even images. However, since machine learning models are mathematical models, they are only able to process numbers. This course will provide you with techniques to transform the different types of data into numbers that can be interpreted by a model.

As you will discover throughout this course, preprocessing is a crucial step in machine learning, and it is a Science in its own right! Data Scientists often spend a lot of time (some statistics say preprocessing accounts for 80% of the total duration of a machine learning project) figuring out how to properly prepare the data to optimize the performance of a machine learning model.

This course introduces the "basic" steps of preprocessing, the ones you will almost always encounter when you do machine learning. You will use two libraries to code the different preprocessing steps in Python: pandas and scikit-learn.


## The different types of data 🔢🔢

### Structured data 📊

All data that can be represented in tabular form is called "structured data" (it is also sometimes called a "flat database"). The purpose of the ETL process is to produce such data: SQL tables, CSV files, dictionaries returned by an API, etc. A simple rule: if you can read and store your data in a pandas DataFrame, then you are working with structured data.

When working with structured data, the columns of the table are called variables: age, salary, gender, country, etc. These variables can be classified into four broad families. These families are useful to keep in mind, because the preprocessing steps depend on the family of the variable we are working on.

#### The four main families of variables 📈

**Quantitative/numeric variables**
These are all the variables that are written as a number. There are two sub-categories:

- continuous quantitative variables: can take any value in a given interval (similar to Python's *float* format). For example: a distance, a temperature, a salary, a patient's blood sugar level, etc.
- discrete quantitative variables: the possible values are separated by gaps of a given length (similar to Python's *integer* format). For example: age, shoe size, exam score, etc.

**Qualitative/categorical variables** 
These are all the variables that are not (directly) expressed as a number, or that take a finite number of possible values. They are expressed as short texts, called "modalities" or categories, or sometimes as a limited set of numbers. There are two subfamilies:

- ordinal qualitative variables: there is a hierarchy between the modalities. For example: the answers to a satisfaction poll: "very bad", "bad", "average", "good", "very good", or house sizes: "small house" / "large house" / "very large house".
- nominal qualitative variables: there is no hierarchy between modalities. For example: nationality, gender, socio-professional category, cities, postcodes... In general, most categorical variables are nominal.

#### See it with data 🔬: the families of variables in the Titanic dataset

The type stored by pandas (`dtype`) is a first hint, but it is not enough: you also need to look at the values. Counting the distinct values of each column helps a lot.

In [ ]:
overview = pd.DataFrame({
    "dtype": titanic.dtypes.astype(str),
    "distinct values": titanic.nunique(),
    "example": titanic.iloc[0],
})
overview

,dtype,distinct values,example
PassengerId,int64,891,1
Survived,int64,2,0
Pclass,int64,3,3
Name,str,891,"Braund, Mr. Owen Harris"
Sex,str,2,male
Age,float64,88,22.0
SibSp,int64,7,1
Parch,int64,7,0
Ticket,str,681,A/5 21171
Fare,float64,248,7.25


Reading this table with the four families in mind:

* `Age` and `Fare` are continuous quantitative variables (`float64`, many distinct values); `SibSp` and `Parch` (number of siblings/spouses and of parents/children on board) are discrete quantitative variables.
* `Sex` and `Embarked` are nominal categorical variables (text, 2 and 3 categories).
* `Pclass` is stored as a number (`int64`), but it is an ordinal categorical variable: 1st, 2nd and 3rd class, only 3 distinct values. The `dtype` alone would have misled us!
* `Survived` (0 or 1) is the natural target variable of this dataset: it is categorical, so predicting it is a classification problem.

#### Target variable / explanatory variables 💹

Most of the time in machine learning, we build models to predict the value of one variable from the other variables: this is called *supervised* machine learning. Supervised machine learning often uses the following notations and vocabulary:

- **Y**, the "target variable", "explained variable" or "predicted variable", is the variable you are trying to predict.
- **X**, the "explanatory variables" are the other variables that the model will use to make its prediction.

For example: if you create a model that predicts a person's salary from their professional experience in years, you are doing supervised machine learning. **Y** is the salary and **X** is the person's experience. To train the model, you need data where the values of both **X** and **Y** are known: training means teaching the model to make the best possible salary prediction based on experience.


### Unstructured data 📋

Unstructured data is any data that cannot be stored in a tabular or JSON/dictionary format. Here are some examples of unstructured data:

- all the articles of Wikipedia (we call this a corpus of texts)
- images
- digitized sounds

The examples listed above require specific preprocessing techniques that you will learn later on during the Deep Learning module. For now, this course focuses on the preparation of structured data.

## The different types of preprocessing for structured data ⚙️⚙️

### Drop rows and columns 🗑️

First of all, we have to clean the dataset to keep only the rows and columns that will be useful for machine learning.

#### Drop columns ⬇️
Some columns need to be excluded from our datasets because they are not suitable to train a machine learning model:

- All columns that are "unique identifiers": dataset index, surname/first name of a person, social security number, transaction number. Machine learning learns by finding common patterns among the explanatory variables, so a variable that takes a different value on each row cannot reveal any pattern.
- Columns with too many missing values. There is no general rule, but if the rate of missing values exceeds 60-70%, it is usually better to exclude the column from the dataset. Missing values cannot be used to train a model, so the rows that contain them must either be excluded (here, 60-70% of all available data!) or be filled with values chosen by the data scientist (introducing a bias in 60-70% of the rows). In both cases, it's usually better not to work with such variables.
- Nominal variables that have too many modalities. Again, there is no general rule, but for the small datasets of this course, you will exclude columns that have more than 20-30 modalities. This rule must be interpreted relative to the size of the dataset: a variable with 100 modalities over 1,000 rows can't be used as is, but the same variable in a sample of 1,000,000 rows could well be used for predictions.
- If two columns are collinear (correlation coefficient equal to 1 or -1), only one of them is kept: typically, you should never keep both the age and the year of birth of a person in your dataset.



::: {.callout-tip title="Intuition"}
A model learns from what rows have **in common**. A passenger ID is like a student number: it distinguishes everyone perfectly, but it tells you nothing about who will pass the exam. And a column that is mostly empty forces you to either throw away most rows or invent most of its values.
:::



#### See it with data 🔬: which columns should we drop?

For each column, let's compute the number of distinct values and the percentage of missing values.

In [ ]:
check = pd.DataFrame({
    "distinct values": titanic.nunique(),
    "% missing": (titanic.isna().mean() * 100).round(1),
})
check.sort_values("% missing", ascending=False)

,distinct values,% missing
Cabin,147,77.1
Age,88,19.9
Embarked,3,0.2
PassengerId,891,0.0
Name,891,0.0
Pclass,3,0.0
Survived,2,0.0
Sex,2,0.0
Parch,7,0.0
SibSp,7,0.0


The table applies the rules above:

* `PassengerId` and `Name` have 891 distinct values for 891 rows: they are unique identifiers, so we drop them. `Ticket` (681 distinct values) is almost an identifier too.
* `Cabin` is missing for 77.1% of the passengers: above the 60-70% threshold, so we drop it.
* `Age` is missing for 19.9% of the passengers: too precious to drop, we will impute it (see below). `Embarked` misses only 0.2% of its values.

Now, the collinearity rule. Let's add a (made-up) birth year computed from the age, and look at the correlation between the two columns.

In [ ]:
demo = titanic[["Age"]].copy()
demo["BirthYear"] = 1912 - demo["Age"]      # the Titanic sank in 1912
print("Correlation between Age and BirthYear:", round(demo["Age"].corr(demo["BirthYear"]), 4))

Correlation between Age and BirthYear: -1.0


The correlation is exactly -1.0: `BirthYear` carries no information that `Age` doesn't already have. Keeping both would add nothing to the model, and it would even prevent some models (like linear regression) from being trained properly, as you will see in the next module.

#### Drop rows ➡️
The following rows are excluded from the dataset: 

- If we are working on a supervised machine learning problem, all the rows where the target variable **Y** is missing.
- The rows with too many outliers or too many missing values, i.e. those with "strange" or even inconsistent values, or values very far from the usual ones: negative age, very high salary compared to the rest of the sample, unknown city name that you only find once in the dataset, ...

#### See it with data 🔬: one outlier can distort a whole column

The small dataset below contains a few suspicious values: a negative age, and a salary of one billion. Let's see what the billion does to the average salary.

In [ ]:
small = pd.read_csv("../data/dataset_preprocessing.csv")
print(small[["Country", "Age", "Salary"]].tail(3))

salary = small["Salary"]
without = salary[salary < 1_000_000]
print(f"\nMean salary  : with the outlier {salary.mean():>14,.0f}   without it {without.mean():>8,.0f}")
print(f"Median salary: with the outlier {salary.median():>14,.0f}   without it {without.median():>8,.0f}")

   Country   Age      Salary
9   France  37.0       67000
10  France -10.0       32000
11   Spain  67.0  1000000000

Mean salary  : with the outlier     83,389,583   without it   61,364
Median salary: with the outlier         64,000   without it   61,000


A single row out of 12 moves the mean salary from 61,364 to 83,389,583: the mean no longer describes anybody in the dataset. The median barely moves (61,000 → 64,000), because it only depends on the order of the values. This is why rows with impossible or extreme values (here a salary of one billion, or an age of -10) are removed before training, and why the median is preferred to the mean for variables with extreme values.

### Imputation of missing values 🔮
Imputation means replacing missing values with valid information. There are many imputation methods, and it is a Science in its own right. Below are the most common ones. Depending on the type of variable, you will use one or the other.

#### Quantitative variables 🔢
**Mean Imputation**

Mean imputation replaces the missing values with the mean value of the variable. This method works on continuous quantitative variables.

**Median Imputation**

Median imputation replaces the missing values with the median of the variable. This method is used for discrete quantitative variables, or for continuous quantitative variables that may have extreme values, for example salary.


#### Categorical variables 🔠
For categorical variables, no mean or median can be calculated. The simplest way is then to replace the missing values with the most frequent modality of the variable. This is called "mode" imputation.
Another very simple method for categorical variables is to replace all missing values with the character string 'missing'. The model then sees a missing value as a distinct piece of information, instead of an existing category.



::: {.callout-tip title="Intuition"}
Imputing is making an educated guess for the empty cells. Mean imputation guesses "a typical passenger", which keeps the average of the column unchanged, but every guess is the same value: the column then looks less spread out than it really is. That's why the share of missing values matters so much.
:::



#### See it with data 🔬: what imputation does to a column

In the Titanic dataset, 177 ages are missing. Let's fill them with the mean, and compare the column before and after.

In [ ]:
age = titanic["Age"]
age_imputed = age.fillna(age.mean())

pd.DataFrame({"before": age.describe(), "after mean imputation": age_imputed.describe()}).round(2)

,before,after mean imputation
count,714.00,891.00
mean,29.70,29.70
std,14.53,13.00
min,0.42,0.42
25%,20.12,22.00
50%,28.00,29.70
75%,38.00,35.00
max,80.00,80.00


In [ ]:
both = pd.concat([pd.DataFrame({"Age": age.dropna(), "version": "before (missing values ignored)"}),
                  pd.DataFrame({"Age": age_imputed, "version": "after mean imputation"})])
fig = px.histogram(both, x="Age", color="version", barmode="overlay", nbins=40,
                   title="Mean imputation creates a spike at the mean age",
                   labels={"Age": "Age (years)", "count": "Number of passengers"})
fig.update_layout(yaxis_title="Number of passengers")
fig.show()

After mean imputation, the mean age is unchanged (29.70) and we now have 891 values instead of 714. But the standard deviation drops from 14.53 to 13.00, and the histogram shows a tall spike at 29.7: 177 passengers now have exactly the same age. The column looks less diverse than reality. With 20% of missing values this is acceptable; with 70% the column would mostly be made of our own guesses.

For a skewed variable such as the ticket price (`Fare`), the mean and the median are very different, and the median is a better "typical value". For a categorical variable such as the port of embarkation (`Embarked`, 2 missing values), we use the most frequent category (the mode).

In [ ]:
fare = titanic["Fare"]
print(f"Fare: mean = {fare.mean():.2f}, median = {fare.median():.2f}")
print(f"Share of passengers who paid less than the mean: {(fare < fare.mean()).mean():.0%}")

print("\nEmbarked:", titanic["Embarked"].value_counts().to_dict(), "| missing:", titanic["Embarked"].isna().sum())
embarked_imputed = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])
print("After mode imputation:", embarked_imputed.value_counts().to_dict())

Fare: mean = 32.20, median = 14.45
Share of passengers who paid less than the mean: 76%

Embarked: {'S': 644, 'C': 168, 'Q': 77} | missing: 2
After mode imputation: {'S': 646, 'C': 168, 'Q': 77}


The mean fare (32.20) is more than twice the median fare (14.45), and 76% of the passengers paid less than the mean: a few very expensive tickets pull the mean up. The median is a much better "typical fare", so median imputation would be the right choice for this column. For `Embarked`, the 2 missing values become `S` (Southampton), by far the most frequent port (644 passengers, then 646 after imputation).

### Standardization/Normalization (quantitative variables only) 📊
To avoid dealing with very large values (this is never advisable in computing, for memory management reasons), and to help our models train properly, we normalize all the quantitative explanatory variables so that their values stay within reasonable intervals, typically a few units. There are various methods to normalize data, but the most common one is called Standardization: the variable $X$ is replaced with $\frac{X - \bar{X}}{\sqrt{Var(X)}}$. We subtract the mean of the variable and divide by its standard deviation, which gives a variable with a mean of 0 and a standard deviation of 1.

Note: in supervised machine learning, the target variable is never normalized. However, it can be rescaled when its distribution is very skewed: in that case, we can apply a $\log$ transformation to the target variable!



::: {.callout-tip title="Intuition"}
Standardization changes the unit of each variable to a common one: "how many standard deviations above or below the average". An age of 44 becomes about +1 (one standard deviation above the average age), whatever the original unit was. The shape of the distribution doesn't change, only its position and its scale.
:::



#### See it with data 🔬: standardization before and after

Let's standardize the age (after mean imputation) and the fare with the formula above.

In [ ]:
numeric = pd.DataFrame({"Age": age_imputed, "Fare": fare})
standardized = (numeric - numeric.mean()) / numeric.std()

pd.concat({"before": numeric.describe().loc[["mean", "std", "min", "max"]],
           "after": standardized.describe().loc[["mean", "std", "min", "max"]]}, axis=1).round(2)

before         after      
        Age    Fare   Age  Fare
mean  29.70   32.20  0.00 -0.00
std   13.00   49.69  1.00  1.00
min    0.42    0.00 -2.25 -0.65
max   80.00  512.33  3.87  9.66

In [ ]:
long = standardized.melt(var_name="variable", value_name="standardized value")
fig = px.histogram(long, x="standardized value", color="variable", barmode="overlay", nbins=60,
                   title="After standardization, Age and Fare share the same scale",
                   labels={"standardized value": "Standardized value (standard deviations from the mean)"})
fig.update_layout(yaxis_title="Number of passengers")
fig.show()

Before standardization, the two columns live on very different scales (ages up to 80, fares up to 512.33). After it, both have a mean of 0 and a standard deviation of 1. Look at the min and max, though: the standardized age goes from -2.25 to 3.87, while the standardized fare reaches 9.66. Standardization does not remove extreme values or change the shape of a distribution: the fare is still very skewed, with a long tail to the right in the histogram.

The note above mentions the $\log$ transformation for skewed variables. `Fare` is a good example of a skewed variable: let's measure its skewness (0 means symmetric) before and after a log transformation (we use `log(1 + x)` because some fares are 0).

In [ ]:
print(f"Skewness of Fare          : {fare.skew():.2f}")
print(f"Skewness of log(1 + Fare) : {np.log1p(fare).skew():.2f}")

Skewness of Fare          : 4.79
Skewness of log(1 + Fare) : 0.39


The log transformation brings the skewness from 4.79 down to 0.39: the long tail of expensive tickets is compressed, and the distribution becomes much more symmetric.

### Encoding (categorical variables only) 💻
Finally, since machine learning models only deal with numbers, all categorical variables must be encoded, that is, turned into numbers. Depending on the type of variable, different methods will be used:


#### Target variable 🎯
In supervised machine learning, we often deal with categorical target variables. In this case, we simply encode the target by mapping each modality to a number. This transformation can be summarized as follows:

|  Y  | 
|-----|
| no  |
| yes |
| no  |
| no  |
| yes |

becomes:

| Y | 
|---|
| 0 |
| 1 |
| 0 |
| 0 |
| 1 |

#### Explanatory variables 🔡
**Ordinal variable**
Ordinal variables can be encoded in the same way as the target variable. If you do so, you may then normalize them as if they were quantitative variables. However, remember that ordinal categorical variables are rare.

**Nominal variable**
Most of the time, this is the method you will use to encode categorical explanatory variables.

A nominal variable cannot be encoded by simply replacing each modality with a different number, because numbers have a hierarchy (2 is bigger than 1), and nominal variables don't. The simplest method is then One Hot Encoding (or Dummy Encoding): we create a binary variable for each modality. Here is an example to make it clearer:

| Country |
|---------|
| France  |
| France  |
| Spain   |
| Germany |
| Germany |
| Spain   |
| Spain   |

becomes:

| France | Spain | Germany |
|--------|-------|---------|
|    1    |   0    |    0     |
|    1    |   0    |    0     |
|    0    |   1    |    0     |
|    0    |   0    |    1     |
|    0    |   0    |    1     |
|    0    |   1    |    0     |
|    0    |   1    |    0     |



::: {.callout-tip title="Intuition"}
If we encoded the ports as Southampton = 1, Cherbourg = 2, Queenstown = 3, the model would believe that Queenstown is "three times" Southampton, or that Cherbourg is "between" the two others. One-hot encoding avoids this: each category gets its own yes/no column, and no category is bigger than another.
:::



An additional subtlety: we learned at the beginning of this course that we should avoid collinear columns. In the above example, if the values in the first two columns are known, the values in the third column can be deduced with the linear equation Germany = 1 - France - Spain: this is called collinearity. For this reason, one of the columns produced by the encoding is usually dropped, which gives: 

| France | Spain |
|--------|-------|
|    1    |   0    |
|    1    |   0    |
|    0    |   1    |
|    0    |   0    |
|    0    |   0    |
|    0    |   1    |
|    0    |   1    |

This way, no information is lost, and collinearity is avoided.

#### See it with data 🔬: one-hot encoding of the port of embarkation

Let's one-hot encode `Embarked` (C = Cherbourg, Q = Queenstown, S = Southampton) with pandas, first with all the columns, then dropping one of them.

In [ ]:
ports = embarked_imputed.head(6)
all_columns = pd.get_dummies(ports, prefix="Embarked", dtype=int)
all_columns["sum of the row"] = all_columns.sum(axis=1)
pd.concat([ports, all_columns], axis=1)

,Embarked,Embarked_C,Embarked_Q,Embarked_S,sum of the row
0,S,0,0,1,1
1,C,1,0,0,1
2,S,0,0,1,1
3,S,0,0,1,1
4,S,0,0,1,1
5,Q,0,1,0,1


In [ ]:
dummies = pd.get_dummies(embarked_imputed, prefix="Embarked", dtype=int)
print("Rows where the 3 columns don't sum to 1:", (dummies.sum(axis=1) != 1).sum())

dropped = pd.get_dummies(embarked_imputed, prefix="Embarked", dtype=int, drop_first=True)
print("Columns kept with drop_first=True:", list(dropped.columns))

Rows where the 3 columns don't sum to 1: 0
Columns kept with drop_first=True: ['Embarked_Q', 'Embarked_S']


Each row has exactly one 1 among the three columns, so the three columns always sum to 1 (0 rows break this rule out of 891). This is the collinearity described above: `Embarked_C = 1 - Embarked_Q - Embarked_S`. With `drop_first=True`, pandas drops the first category (`Embarked_C`) and keeps `Embarked_Q` and `Embarked_S`: a passenger with 0 in both columns embarked in Cherbourg, so no information is lost.

::: {.sb-next}
**[Next: Preprocessing with scikit-learn →](../lessons/preprocessing-with-scikit-learn.ipynb)**
:::